# Chapter 3: Prompt Engineering for BA & QA

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Prompt Library Builder** -- a reusable collection of tested, versioned prompts for common BA/QA tasks. Learn techniques like few-shot prompting, chain-of-thought, and structured output.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Completed Chapter 2 lab


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Zero-Shot vs Few-Shot Prompting

Compare how providing examples dramatically improves output quality for BA/QA tasks.


In [ ]:
# Zero-shot: No examples provided
zero_shot = """Write a user story for a shopping cart checkout feature."""

# Few-shot: Provide examples of the desired format
few_shot = """Write a user story for a shopping cart checkout feature.

Follow this exact format, as shown in these examples:

Example 1:
STORY-101: Password Reset
As a registered user
I want to reset my password via email
So that I can regain access if I forget my credentials

Acceptance Criteria:
- Given I click 'Forgot Password', When I enter my registered email, Then I receive a reset link within 2 minutes
- Given I click the reset link, When I enter a new valid password, Then my password is updated
- Given the reset link is older than 24 hours, When I click it, Then I see an expiry message

Priority: High | Story Points: 3 | Sprint: 2024-Q1-S3

Example 2:
STORY-102: Order History
As a returning customer
I want to view my past orders
So that I can reorder items or track deliveries

Acceptance Criteria:
- Given I am logged in, When I navigate to Order History, Then I see orders sorted by date descending
- Given I click an order, When the detail page loads, Then I see items, quantities, and total
- Given I have no past orders, When I visit Order History, Then I see a friendly empty state

Priority: Medium | Story Points: 2 | Sprint: 2024-Q1-S4

Now write a similar user story for the checkout feature:"""

for label, prompt in [('ZERO-SHOT', zero_shot), ('FEW-SHOT', few_shot)]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3
    )
    print(f'\n{"="*60}')
    print(f'{label} RESULT:')
    print(f'{"="*60}')
    print(response.choices[0].message.content)

## 2. Chain-of-Thought for Complex Analysis

Use step-by-step reasoning to improve the LLM's analysis of requirements.


In [ ]:
# Chain-of-thought prompt for requirements analysis
cot_prompt = """Analyze the following requirement for potential issues. 
Think step by step:

Step 1: Check for ambiguity -- are there vague terms?
Step 2: Check for completeness -- what's missing?
Step 3: Check for testability -- can each clause be verified?
Step 4: Check for conflicts -- does it contradict common patterns?
Step 5: Provide your final assessment with specific recommendations.

Requirement: "The system should quickly process all user requests and 
display appropriate error messages when something goes wrong. The data 
should be stored securely and the system must be available most of the time."

Show your reasoning for each step."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': cot_prompt}],
    temperature=0.3
)
print(response.choices[0].message.content)

## 3. Building a Reusable Prompt Library

Create a structured, versioned prompt library for your BA/QA team.


In [ ]:
# Prompt Library with metadata and versioning
prompt_library = {
    'user_story_generator': {
        'version': '1.2',
        'category': 'BA',
        'description': 'Generate user stories with acceptance criteria',
        'system_prompt': 'You are a senior business analyst. Write user stories in standard format with Given/When/Then acceptance criteria.',
        'template': 'Write a user story for: {feature_description}\n\nContext: {project_context}\n\nInclude: story ID, role, goal, benefit, acceptance criteria (3+), priority, and story points.',
        'parameters': {'temperature': 0.3, 'max_tokens': 800}
    },
    'test_case_generator': {
        'version': '1.0',
        'category': 'QA',
        'description': 'Generate test cases from requirements',
        'system_prompt': 'You are a senior QA engineer. Generate comprehensive test cases covering positive, negative, boundary, and edge cases.',
        'template': 'Generate test cases for:\n{requirement}\n\nFormat each test case with: ID, Title, Preconditions, Steps, Expected Result, Priority, Type (Functional/Boundary/Negative/Edge).',
        'parameters': {'temperature': 0.2, 'max_tokens': 1200}
    },
    'requirements_reviewer': {
        'version': '1.1',
        'category': 'BA',
        'description': 'Review requirements for quality issues',
        'system_prompt': 'You are a requirements quality expert. Analyze requirements for ambiguity, completeness, testability, consistency, and feasibility.',
        'template': 'Review this requirement:\n{requirement}\n\nProvide: quality score (1-10), issues found, severity of each issue, and rewritten improved version.',
        'parameters': {'temperature': 0.2, 'max_tokens': 1000}
    }
}

def run_prompt(prompt_name: str, variables: dict) -> str:
    """Execute a prompt from the library with given variables."""
    p = prompt_library[prompt_name]
    user_content = p['template'].format(**variables)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': p['system_prompt']},
            {'role': 'user', 'content': user_content}
        ],
        **p['parameters']
    )
    return response.choices[0].message.content

# Test the prompt library
result = run_prompt('user_story_generator', {
    'feature_description': 'two-factor authentication for mobile app',
    'project_context': 'Banking app with 50K daily active users'
})
print(result)

## Exercises
1. Add a `defect_report_writer` prompt to the library and test it with a sample bug.
2. Experiment with system prompts -- how does changing the persona affect output quality?
3. Create a prompt that uses XML tags to structure the output (e.g., `<requirement>`, `<test_case>`).
4. Build an A/B testing function that compares two prompt versions on the same input.
